<a href="https://colab.research.google.com/github/anhpdd/ml-property-valuation-klang-valley/blob/main/notebooks/2_2_Ridership_data_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd

# Load configuration
from config_loader import SUPPORT_DATA_DIR

In [ ]:
# Load and filter data
df = (
    pd.read_parquet("https://storage.data.gov.my/dashboards/prasarana_timeseries.parquet")
    .assign(date=lambda x: pd.to_datetime(x['date']))
    .query("date >= '2022-01-01'")
    .assign(year_month=lambda x: x['date'].dt.strftime('%Y-%m'))
)

# Aggregate monthly ridership
monthly_ridership = (
    df.groupby(['year_month', 'service', 'origin', 'destination'], as_index=False)
    ['passengers']
    .sum()
    .rename(columns={'year_month': 'date'})
    [['date', 'service', 'origin', 'destination', 'passengers']]
)

print(f"✅ Loaded {len(monthly_ridership):,} monthly ridership records")
monthly_ridership

✅ Loaded 666,396 monthly ridership records


,date,service,origin,destination,passengers
0,2023-01,rail,A0: All Stations,AG01: Sentul Timur,112237
1,2023-01,rail,A0: All Stations,AG02: Sentul,98256
2,2023-01,rail,A0: All Stations,AG03: Titiwangsa,95541
3,2023-01,rail,A0: All Stations,AG04: PWTC,141960
4,2023-01,rail,A0: All Stations,AG05: Sultan Ismail,40720
...,...,...,...,...,...
666391,2025-12,rail,SP31: Putra Heights,SP26: Taman Perindustrian Puchong,556
666392,2025-12,rail,SP31: Putra Heights,SP27: Bandar Puteri,722
666393,2025-12,rail,SP31: Putra Heights,SP28: Puchong Perdana,896
666394,2025-12,rail,SP31: Putra Heights,SP29: Puchong Prima,2192


In [2]:
# Filter for station aggregates
all_station = monthly_ridership.loc[
    (monthly_ridership['origin'] == 'A0: All Stations') |
    (monthly_ridership['destination'] == 'A0: All Stations')
].reset_index(drop=True)

# Outgoing dataframe
outgoing = (
    all_station
    .groupby(['origin', 'date'], as_index=False)
    ['passengers'].sum()
    .rename(columns={'origin': 'station_name', 'passengers': 'outgoing'})
)

# Incoming dataframe
incoming = (
    all_station
    .groupby(['destination', 'date'], as_index=False)
    ['passengers'].sum()
    .rename(columns={'destination': 'station_name', 'passengers': 'incoming'})
)

# Total ridership dataframe
ridership = (
    outgoing
    .merge(incoming, on=['station_name', 'date'], how='left')
    .fillna(0)
    .astype({'outgoing': int, 'incoming': int})
)

ridership

,station_name,date,outgoing,incoming
0,A0: All Stations,2023-01,12763943,12763943
1,A0: All Stations,2023-02,12322923,12322923
2,A0: All Stations,2023-03,14814863,14814863
3,A0: All Stations,2023-04,12329434,12329434
4,A0: All Stations,2023-05,14029921,14029921
...,...,...,...,...
5503,SP31: Putra Heights,2025-08,103758,112139
5504,SP31: Putra Heights,2025-09,96263,103615
5505,SP31: Putra Heights,2025-10,145157,157433
5506,SP31: Putra Heights,2025-11,197552,213146


In [3]:
# Export dataset
ridership.to_csv(SUPPORT_DATA_DIR / 'ridership_data.csv')